In [4]:
import torch
import torch.nn as nn
from einops import rearrange, repeat
from tab_transformer_pytorch import TabTransformer as BaseTabTransformer

import pandas as pd
df_train = pd.read_csv("/home/UT_shared/data/train.csv")
df_val = pd.read_csv("/home/UT_shared/data/val.csv")
df_test = pd.read_csv("/home/UT_shared/data/test.csv")
df_cal = pd.read_csv("/home/UT_shared/data/cal.csv")


In [5]:
import torch
import torch.nn as nn
from einops import rearrange, repeat
from tab_transformer_pytorch import TabTransformer as BaseTabTransformer


class TabTransformerWithEmbedding(BaseTabTransformer):
    def forward(self, x_categ, x_cont, return_embedding=False, return_attn=False):
        xs = []

        assert x_categ.shape[-1] == self.num_categories, (
            f"Expected {self.num_categories} categorical columns, "
            f"but got {x_categ.shape[-1]}"
        )

        # ----- categorical part -----
        if self.num_unique_categories > 0:
            # Newer tab_transformer_pytorch uses self.categorical_embeds
            # x_categ should already be encoded as 0, 1, ..., K-1 for each column.
            is_special_token = x_categ < 0

            categ_embed = self.categorical_embeds(
                x_categ.clamp_min(0),
                sum_discrete_sets=False
            )

            # Handle special negative tokens if present
            if is_special_token.any():
                special_token_ids = (x_categ + 1).abs().clamp_max(
                    self.num_special_tokens - 1
                )
                special_embed = self.special_token_embed(special_token_ids)
                categ_embed = torch.where(
                    is_special_token[..., None],
                    special_embed,
                    categ_embed
                )

            if self.use_shared_categ_embed:
                shared_categ_embed = repeat(
                    self.shared_category_embed,
                    "n d -> b n d",
                    b=categ_embed.shape[0]
                )
                categ_embed = torch.cat((categ_embed, shared_categ_embed), dim=-1)

            if return_attn:
                x, attns = self.transformer(categ_embed, return_attn=True)
            else:
                x = self.transformer(categ_embed)

            flat_categ = rearrange(x, "b ... -> b (...)")
            xs.append(flat_categ)

        # ----- continuous part -----
        assert x_cont.shape[-1] == self.num_continuous, (
            f"Expected {self.num_continuous} continuous columns, "
            f"but got {x_cont.shape[-1]}"
        )

        if self.num_continuous > 0:
            if self.continuous_mean_std is not None:
                mean, std = self.continuous_mean_std.unbind(dim=-1)
                x_cont = (x_cont - mean) / std

            normed_cont = self.norm(x_cont)
            xs.append(normed_cont)

        # ----- joint embedding -----
        x = torch.cat(xs, dim=-1)

        if return_embedding:
            if return_attn:
                return x, attns
            return x

        logits = self.mlp(x)

        if return_attn:
            return logits, attns

        return logits

In [2]:

#device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

num_categ = x_cat_train.shape[1]
num_cont  = x_cont_train.shape[1]       

categories = (2,) * num_categ          

model = TabTransformerWithEmbedding(
    categories      = categories,
    num_continuous  = x_cont_train.shape[1],
    dim             = 32,
    dim_out         = 1,                 # not used when return_embedding=True
    depth           = 6,
    heads           = 8,
    attn_dropout    = 0.1,
    ff_dropout      = 0.1,
    mlp_hidden_mults= (4, 2),
    mlp_act         = nn.ReLU()
)

model.to(device)
model.eval()

TabTransformerWithEmbedding(
  (categorical_embeds): Embed(
    (embeddings): Embedding(122, 28)
    (selectors): ModuleList(
      (0): DiscreteContinuousSelector(
        (discrete_selector): DiscreteSelector(
          (embeddings): Embedding(122, 28)
        )
      )
    )
  )
  (norm): LayerNorm((17,), eps=1e-05, elementwise_affine=True)
  (special_token_embed): Embedding(2, 28)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x ModuleList(
        (0): HyperConnections(
          (branch): PreNorm(
            (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=32, out_features=384, bias=False)
              (to_out): Linear(in_features=128, out_features=32, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (act): Tanh()
          (split_fracs): Rearrange('b ... (f d) -> b ... f d', f=1)
          (merge_fracs): Rearrange('b .

In [3]:
x_cat_train, x_cont_train = x_cat_train.to(device), x_cont_train.to(device)
x_cat_val,   x_cont_val   = x_cat_val.to(device),   x_cont_val.to(device)
x_cat_test,  x_cont_test  = x_cat_test.to(device),  x_cont_test.to(device)
x_cat_cal,  x_cont_cal  = x_cat_cal.to(device),  x_cont_cal.to(device)

with torch.no_grad():
    emb_train = model(x_cat_train, x_cont_train, return_embedding=True).cpu()
    emb_val   = model(x_cat_val,   x_cont_val,   return_embedding=True).cpu()
    emb_test  = model(x_cat_test,  x_cont_test,  return_embedding=True).cpu()
    emb_cal  = model(x_cat_cal,  x_cont_cal,  return_embedding=True).cpu()


AttributeError: 'TabTransformerWithEmbedding' object has no attribute 'categories_offset'

In [13]:
pd.DataFrame(emb_train.numpy()).to_csv("/home/UT_shared/data/emb_train.csv", index=False)
pd.DataFrame(emb_val.numpy()).to_csv("/home/UT_shared/data/emb_val.csv", index=False)
pd.DataFrame(emb_test.numpy()).to_csv("/home/UT_shared/data/emb_test.csv", index=False)


In [16]:
import os
# embeddings of bootstrapped test data
boot_dir = "/home/UT_shared/data/bootstrap500"
out_dir  = os.path.join(boot_dir, "embeddings")
os.makedirs(out_dir, exist_ok=True)

num_boot = 50

for b in range(1, num_boot + 1):
    csv_path = os.path.join(boot_dir, f"bootstrap_{b}.csv")
    print(f"Processing {csv_path}")

    df_boot = pd.read_csv(csv_path)

    # assume same schema as train: we use same column indices
    x_cat = torch.tensor(df_boot.iloc[:, categ_idx].to_numpy(), dtype=torch.long, device=device)
    x_cont = torch.tensor(df_boot.iloc[:, cont_idx].to_numpy(), dtype=torch.float32, device=device)

    with torch.no_grad():
        emb = model(x_cat, x_cont, return_embedding=True).cpu()  # [N_b, D]

    emb_df = pd.DataFrame(emb.numpy())
    # if you have an ID column, you can add it here:
    # emb_df.insert(0, "patient_id", df_boot["patient_id"].values)

    out_path = os.path.join(out_dir, f"emb_bootstrap_{b}.csv")
    emb_df.to_csv(out_path, index=False)
    print(f"Saved embeddings to {out_path}")

Processing /home/UT_shared/data/bootstrap500/bootstrap_1.csv
Saved embeddings to /home/UT_shared/data/bootstrap500/embeddings/emb_bootstrap_1.csv
Processing /home/UT_shared/data/bootstrap500/bootstrap_2.csv
Saved embeddings to /home/UT_shared/data/bootstrap500/embeddings/emb_bootstrap_2.csv
Processing /home/UT_shared/data/bootstrap500/bootstrap_3.csv
Saved embeddings to /home/UT_shared/data/bootstrap500/embeddings/emb_bootstrap_3.csv
Processing /home/UT_shared/data/bootstrap500/bootstrap_4.csv
Saved embeddings to /home/UT_shared/data/bootstrap500/embeddings/emb_bootstrap_4.csv
Processing /home/UT_shared/data/bootstrap500/bootstrap_5.csv
Saved embeddings to /home/UT_shared/data/bootstrap500/embeddings/emb_bootstrap_5.csv
Processing /home/UT_shared/data/bootstrap500/bootstrap_6.csv
Saved embeddings to /home/UT_shared/data/bootstrap500/embeddings/emb_bootstrap_6.csv
Processing /home/UT_shared/data/bootstrap500/bootstrap_7.csv
Saved embeddings to /home/UT_shared/data/bootstrap500/embedding

In [8]:
import torch
import torch.nn as nn
from einops import rearrange, repeat
from tab_transformer_pytorch import TabTransformer as BaseTabTransformer

import pandas as pd
import numpy as np


# ============================================================
# 1. Load data
# ============================================================

df_train = pd.read_csv("/home/UT_shared/data/train.csv")
df_val   = pd.read_csv("/home/UT_shared/data/val.csv")
df_test  = pd.read_csv("/home/UT_shared/data/test.csv")
df_cal   = pd.read_csv("/home/UT_shared/data/cal.csv")


# ============================================================
# 2. Define categorical and continuous column indices
# ============================================================

categ_idx = [
    1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,
    17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30,
    31, 32, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47,
    60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73,
    74, 79, 75, 76
]

cont_idx = [
    0, 33, 34, 35, 36,
    48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59
]


# ============================================================
# 3. Preprocess categorical and continuous variables
# ============================================================

def preprocess_dataframes(dataframes, categ_idx, cont_idx):
    """
    Preprocess categorical and continuous columns.

    Categorical:
        - Fill missing values with 0
        - Convert to integer
        - If values are coded as {1, 2}, convert to {0, 1}

    Continuous:
        - Convert to numeric
        - Fill missing values using train-set median
    """

    df_train = dataframes[0]

    # ---------- categorical preprocessing ----------
    for df in dataframes:
        for col_idx in categ_idx:
            col = df.columns[col_idx]

            df[col] = df[col].fillna(0)
            df[col] = df[col].astype(int)

            unique_vals = set(df[col].dropna().unique())

            # Convert 1/2 coding to 0/1 coding
            if unique_vals.issubset({1, 2}):
                df[col] = df[col] - 1

    # ---------- continuous preprocessing ----------
    train_medians = {}

    for col_idx in cont_idx:
        col = df_train.columns[col_idx]

        df_train[col] = pd.to_numeric(df_train[col], errors="coerce")
        train_medians[col] = df_train[col].median()

    for df in dataframes:
        for col_idx in cont_idx:
            col = df.columns[col_idx]

            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].fillna(train_medians[col])

    return dataframes


df_train, df_val, df_test, df_cal = preprocess_dataframes(
    [df_train, df_val, df_test, df_cal],
    categ_idx=categ_idx,
    cont_idx=cont_idx
)


# ============================================================
# 4. Check categorical coding
# ============================================================

print("Checking categorical variables...")

for j, col_idx in enumerate(categ_idx):
    col = df_train.columns[col_idx]
    vals = sorted(df_train[col].dropna().unique())

    print(f"Categorical variable {j}: column index={col_idx}, name={col}, values={vals}")

    if min(vals) < 0:
        raise ValueError(f"Column {col} contains negative values after preprocessing.")

    if max(vals) > 1:
        raise ValueError(
            f"Column {col} has values larger than 1: {vals}. "
            "But categories=(2,) assumes binary variables coded as 0/1."
        )


# ============================================================
# 5. Convert data to tensors
# ============================================================

x_cat_train = torch.tensor(df_train.iloc[:, categ_idx].to_numpy(), dtype=torch.long)
x_cont_train = torch.tensor(df_train.iloc[:, cont_idx].to_numpy(), dtype=torch.float32)

x_cat_val = torch.tensor(df_val.iloc[:, categ_idx].to_numpy(), dtype=torch.long)
x_cont_val = torch.tensor(df_val.iloc[:, cont_idx].to_numpy(), dtype=torch.float32)

x_cat_test = torch.tensor(df_test.iloc[:, categ_idx].to_numpy(), dtype=torch.long)
x_cont_test = torch.tensor(df_test.iloc[:, cont_idx].to_numpy(), dtype=torch.float32)

x_cat_cal = torch.tensor(df_cal.iloc[:, categ_idx].to_numpy(), dtype=torch.long)
x_cont_cal = torch.tensor(df_cal.iloc[:, cont_idx].to_numpy(), dtype=torch.float32)


# ============================================================
# 6. Define TabTransformer with embedding extraction
# ============================================================

class TabTransformerWithEmbedding(BaseTabTransformer):
    def forward(self, x_categ, x_cont, return_embedding=False, return_attn=False):
        xs = []

        # ====================================================
        # categorical part
        # ====================================================

        assert x_categ.shape[-1] == self.num_categories, (
            f"Expected {self.num_categories} categorical columns, "
            f"but got {x_categ.shape[-1]}."
        )

        if self.num_unique_categories > 0:
            # Newer tab_transformer_pytorch uses self.categorical_embeds
            # instead of self.categories_offset and self.category_embed.

            is_special_token = x_categ < 0

            categ_embed = self.categorical_embeds(
                x_categ.clamp_min(0),
                sum_discrete_sets=False
            )

            # Only needed if negative special tokens exist
            if is_special_token.any():
                special_token_ids = (x_categ + 1).abs().clamp_max(
                    self.num_special_tokens - 1
                )

                special_embed = self.special_token_embed(special_token_ids)

                categ_embed = torch.where(
                    is_special_token[..., None],
                    special_embed,
                    categ_embed
                )

            if self.use_shared_categ_embed:
                shared_categ_embed = repeat(
                    self.shared_category_embed,
                    "n d -> b n d",
                    b=categ_embed.shape[0]
                )

                categ_embed = torch.cat(
                    (categ_embed, shared_categ_embed),
                    dim=-1
                )

            if return_attn:
                x, attns = self.transformer(categ_embed, return_attn=True)
            else:
                x = self.transformer(categ_embed)

            flat_categ = rearrange(x, "b ... -> b (...)")
            xs.append(flat_categ)

        # ====================================================
        # continuous part
        # ====================================================

        assert x_cont.shape[-1] == self.num_continuous, (
            f"Expected {self.num_continuous} continuous columns, "
            f"but got {x_cont.shape[-1]}."
        )

        if self.num_continuous > 0:
            if self.continuous_mean_std is not None:
                mean, std = self.continuous_mean_std.unbind(dim=-1)
                x_cont = (x_cont - mean) / std

            normed_cont = self.norm(x_cont)
            xs.append(normed_cont)

        # ====================================================
        # joint representation
        # ====================================================

        x = torch.cat(xs, dim=-1)

        if return_embedding:
            if return_attn:
                return x, attns
            return x

        logits = self.mlp(x)

        if return_attn:
            return logits, attns

        return logits


# ============================================================
# 7. Initialize model
# ============================================================

device = "cpu"

num_categ = x_cat_train.shape[1]
num_cont = x_cont_train.shape[1]

# Since all categorical variables are binary after preprocessing
categories = (2,) * num_categ

model = TabTransformerWithEmbedding(
    categories=categories,
    num_continuous=num_cont,
    dim=32,
    dim_out=1,
    depth=6,
    heads=8,
    attn_dropout=0.1,
    ff_dropout=0.1,
    mlp_hidden_mults=(4, 2),
    mlp_act=nn.ReLU()
)

model = model.to(device)
model.eval()


# ============================================================
# 8. Move tensors to device
# ============================================================

x_cat_train = x_cat_train.to(device)
x_cont_train = x_cont_train.to(device)

x_cat_val = x_cat_val.to(device)
x_cont_val = x_cont_val.to(device)

x_cat_test = x_cat_test.to(device)
x_cont_test = x_cont_test.to(device)

x_cat_cal = x_cat_cal.to(device)
x_cont_cal = x_cont_cal.to(device)


# ============================================================
# 9. Extract embeddings
# ============================================================

with torch.no_grad():
    emb_train = model(
        x_cat_train,
        x_cont_train,
        return_embedding=True
    ).cpu()

    emb_val = model(
        x_cat_val,
        x_cont_val,
        return_embedding=True
    ).cpu()

    emb_test = model(
        x_cat_test,
        x_cont_test,
        return_embedding=True
    ).cpu()

    emb_cal = model(
        x_cat_cal,
        x_cont_cal,
        return_embedding=True
    ).cpu()


# ============================================================
# 10. Check embedding shapes
# ============================================================

print("Embedding shapes:")
print("emb_train:", emb_train.shape)
print("emb_val:  ", emb_val.shape)
print("emb_test: ", emb_test.shape)
print("emb_cal:  ", emb_cal.shape)


# ============================================================
# 11. Save embeddings as CSV
# ============================================================

def save_embedding_csv(embedding_tensor, output_path):
    emb_np = embedding_tensor.numpy()

    emb_df = pd.DataFrame(
        emb_np,
        columns=[f"emb_{i}" for i in range(emb_np.shape[1])]
    )

    emb_df.to_csv(output_path, index=False)


save_embedding_csv(emb_train, "/home/UT_shared/data/emb_train.csv")
save_embedding_csv(emb_val,   "/home/UT_shared/data/emb_val.csv")
save_embedding_csv(emb_test,  "/home/UT_shared/data/emb_test.csv")
save_embedding_csv(emb_cal,   "/home/UT_shared/data/emb_cal.csv")

print("Embeddings saved successfully.")

Checking categorical variables...
Categorical variable 0: column index=1, name=hypertension_icd10, values=[np.int64(0), np.int64(1)]
Categorical variable 1: column index=2, name=diabetes_combined, values=[np.int64(0), np.int64(1)]
Categorical variable 2: column index=3, name=dyslipidemia_combined, values=[np.int64(0), np.int64(1)]
Categorical variable 3: column index=4, name=dcm_icd10, values=[np.int64(0), np.int64(1)]
Categorical variable 4: column index=5, name=hcm_icd10, values=[np.int64(0), np.int64(1)]
Categorical variable 5: column index=6, name=myocarditis_icd10_prior, values=[np.int64(0), np.int64(1)]
Categorical variable 6: column index=7, name=pericarditis_icd10_prior, values=[np.int64(0), np.int64(1)]
Categorical variable 7: column index=8, name=aortic_aneurysm_icd10, values=[np.int64(0), np.int64(1)]
Categorical variable 8: column index=9, name=aortic_dissection_icd10_prior, values=[np.int64(0), np.int64(1)]
Categorical variable 9: column index=10, name=pulmonary_htn_icd10,